In [3]:
import os
os.environ["PYTHONWARNINGS"] = "ignore"

import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    cross_val_score
)
from sklearn.metrics import (
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

import xgboost as xgb
from catboost import CatBoostClassifier


# ======================
# 输出文件夹
# ======================
out_dir = "ml_figures_nested_cv_9models"
os.makedirs(out_dir, exist_ok=True)


# ======================
# 读取数据
# ======================
print("=== Step 1/8: 读取数据 ===", flush=True)
df = pd.read_csv("df_expr.csv")

genes = ["ZFP36L2", "LCORL", "SFMBT2"]

missing_genes = [g for g in genes if g not in df.columns]
if missing_genes:
    raise ValueError(f"以下基因列不存在: {missing_genes}")

if "label" not in df.columns:
    raise ValueError("df_expr.csv 中缺少 'label' 列")

X = df[genes].apply(pd.to_numeric, errors="coerce").fillna(0.0)
y = pd.to_numeric(df["label"], errors="coerce")

if y.isna().any():
    raise ValueError("label 列存在非数值或缺失值，请检查数据")

y = y.astype(int)

if set(np.unique(y)) - {0, 1}:
    raise ValueError("label 必须为二分类编码 0/1")

print("数据读取完成。", flush=True)


# ======================
# 颜色设置
# ======================
# Figure F 的3个基因颜色（保持你之前要求）
single_gene_colors = ["#d88f91", "#b8b8bb", "#80bcc8"]

# Figure G / H / I 的9个模型颜色
model_names = ["KNN", "LDA", "LR", "NB", "RF", "RPART", "SVM", "XGBoost", "CatBoost"]
model_colors = [
    "#FFADAD", "#FFD6A5", "#FDFFB6",
    "#CAFFBF", "#9BF6FF", "#A0C4FF",
    "#BDB2FF", "#FFC6FF", "#E0BBE4"
]
color_map = dict(zip(model_names, model_colors))


# ======================
# Figure F 单基因 ROC
# ======================
print("=== Step 2/8: 绘制 Figure F 单基因 ROC ===", flush=True)
plt.figure(figsize=(6, 6))

for g, color in zip(genes, single_gene_colors):
    fpr, tpr, _ = roc_curve(y, X[g])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{g} AUC={roc_auc:.3f}", color=color, linewidth=2)

plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Single Gene ROC")
plt.legend()
plt.tight_layout()
plt.savefig(f"{out_dir}/Figure_F_single_gene_ROC.svg", dpi=300)
plt.close()
print("Figure F 已保存。", flush=True)


# ======================
# 80%训练 / 20%独立测试
# ======================
print("=== Step 3/8: 划分训练集和测试集 ===", flush=True)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
print("数据划分完成。", flush=True)


# ======================
# 九种模型 + 参数网格
# ======================
model_configs = {
    "KNN": {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier())
        ]),
        "param_grid": {
            "model__n_neighbors": [3, 5, 7, 9],
            "model__weights": ["uniform", "distance"]
        }
    },

    "LDA": {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LinearDiscriminantAnalysis())
        ]),
        "param_grid": {
            "model__solver": ["svd", "lsqr"]
        }
    },

    "LR": {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, random_state=42))
        ]),
        "param_grid": {
            "model__C": [0.01, 0.1, 1, 10],
            "model__solver": ["liblinear", "lbfgs"]
        }
    },

    "NB": {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("model", GaussianNB())
        ]),
        "param_grid": {
            "model__var_smoothing": [1e-9, 1e-8, 1e-7]
        }
    },

    "RF": {
        "pipeline": Pipeline([
            ("model", RandomForestClassifier(random_state=42))
        ]),
        "param_grid": {
            "model__n_estimators": [100, 200, 300],
            "model__max_depth": [None, 3, 5, 10],
            "model__min_samples_split": [2, 5],
            "model__class_weight": [None, "balanced"]
        }
    },

    "RPART": {
        "pipeline": Pipeline([
            ("model", DecisionTreeClassifier(random_state=42))
        ]),
        "param_grid": {
            "model__max_depth": [None, 3, 5, 10],
            "model__min_samples_split": [2, 5, 10],
            "model__class_weight": [None, "balanced"]
        }
    },

    "SVM": {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVC(probability=True, random_state=42))
        ]),
        "param_grid": {
            "model__C": [0.1, 1, 10],
            "model__kernel": ["linear", "rbf"],
            "model__gamma": ["scale", "auto"]
        }
    },

    "XGBoost": {
        "pipeline": Pipeline([
            ("model", xgb.XGBClassifier(
                eval_metric="logloss",
                random_state=42,
                verbosity=0
            ))
        ]),
        "param_grid": {
            "model__n_estimators": [100, 200],
            "model__max_depth": [2, 3, 5],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__subsample": [0.8, 1.0]
        }
    },

    "CatBoost": {
        "pipeline": Pipeline([
            ("model", CatBoostClassifier(
                verbose=0,
                random_state=42
            ))
        ]),
        "param_grid": {
            "model__depth": [3, 4, 6],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__iterations": [100, 200, 300]
        }
    }
}


# ======================
# 内部5折调参 + 外部10折评估
# ======================
print("=== Step 4/8: 模型训练与评估开始 ===", flush=True)
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

benchmark_records = []
best_estimators = {}
best_params_summary = []

total_models = len(model_configs)

for idx, (name, config) in enumerate(model_configs.items(), start=1):
    start_time = time.time()
    print(f"\n[{idx}/{total_models}] 正在处理模型: {name} ...", flush=True)

    grid = GridSearchCV(
        estimator=config["pipeline"],
        param_grid=config["param_grid"],
        scoring="roc_auc",
        cv=inner_cv,
        n_jobs=-1,
        refit=True,
        verbose=0
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    best_estimators[name] = best_model

    best_params_summary.append({
        "model": name,
        "best_params": str(grid.best_params_),
        "best_inner_cv_auc": grid.best_score_
    })

    outer_auc_scores = cross_val_score(
        best_model,
        X_train,
        y_train,
        cv=outer_cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    for score in outer_auc_scores:
        benchmark_records.append([name, score])

    elapsed = time.time() - start_time
    print(f"[{idx}/{total_models}] {name} 完成", flush=True)
    print(f"  最佳参数: {grid.best_params_}", flush=True)
    print(f"  内部5折最佳AUC: {grid.best_score_:.4f}", flush=True)
    print(f"  外部10折平均AUC: {outer_auc_scores.mean():.4f}", flush=True)
    print(f"  用时: {elapsed:.2f} 秒", flush=True)

print("\n所有模型训练完成。", flush=True)

benchmark_df = pd.DataFrame(benchmark_records, columns=["model", "AUC"])
benchmark_df.to_csv(f"{out_dir}/benchmark_auc_scores.csv", index=False)

best_params_df = pd.DataFrame(best_params_summary)
best_params_df.to_csv(f"{out_dir}/best_params_summary.csv", index=False)


# ======================
# Figure G Benchmark boxplot
# ======================
print("=== Step 5/8: 绘制 Figure G Benchmark 箱线图 ===", flush=True)
plt.figure(figsize=(10, 5))

data_to_plot = [benchmark_df.loc[benchmark_df["model"] == m, "AUC"] for m in model_names]
box = plt.boxplot(data_to_plot, labels=model_names, patch_artist=True)

for patch, m in zip(box["boxes"], model_names):
    patch.set_facecolor(color_map[m])

for whisker, m in zip(box["whiskers"], [m for m in model_names for _ in (0, 1)]):
    whisker.set_color(color_map[m])

for cap, m in zip(box["caps"], [m for m in model_names for _ in (0, 1)]):
    cap.set_color(color_map[m])

for median in box["medians"]:
    median.set_color("black")

plt.title("Machine Learning Benchmark (Outer 10-fold CV)")
plt.ylabel("AUC")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f"{out_dir}/Figure_G_benchmark.svg", dpi=300)
plt.close()
print("Figure G 已保存。", flush=True)


# ======================
# 选平均AUC最高的模型
# ======================
print("=== Step 6/8: 选择最佳模型 ===", flush=True)
mean_auc_df = (
    benchmark_df.groupby("model", as_index=False)["AUC"]
    .mean()
    .sort_values("AUC", ascending=False)
)

mean_auc_df.to_csv(f"{out_dir}/benchmark_mean_auc_summary.csv", index=False)

best_name = mean_auc_df.iloc[0]["model"]
best_cv_auc = mean_auc_df.iloc[0]["AUC"]
best_model = best_estimators[best_name]

print(f"最佳模型: {best_name}", flush=True)
print(f"平均CV AUC: {best_cv_auc:.4f}", flush=True)

best_model.fit(X_train, y_train)


# ======================
# Figure H 所有模型在独立测试集上的 ROC
# ======================
print("=== Step 7/8: 绘制 Figure H 和 Figure I ===", flush=True)
plt.figure(figsize=(6, 6))

test_summary = []

for name in model_names:
    print(f"  正在绘制 ROC: {name}", flush=True)
    model = best_estimators[name]
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{name} AUC={roc_auc:.3f}", color=color_map[name], linewidth=2)

    test_summary.append({
        "model": name,
        "test_auc": roc_auc,
        "test_ap": average_precision_score(y_test, prob)
    })

plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves on Independent Test Set")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{out_dir}/Figure_H_ROC_models.svg", dpi=300)
plt.close()
print("Figure H 已保存。", flush=True)


# ======================
# Figure I 所有模型在独立测试集上的 PR
# ======================
plt.figure(figsize=(6, 6))

for name in model_names:
    print(f"  正在绘制 PR: {name}", flush=True)
    model = best_estimators[name]
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)
    plt.plot(recall, precision, label=f"{name} AP={ap:.3f}", color=color_map[name], linewidth=2)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("PR Curves on Independent Test Set")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{out_dir}/Figure_I_PR_models.svg", dpi=300)
plt.close()
print("Figure I 已保存。", flush=True)


# ======================
# Figure J 最佳模型独立测试集 ROC
# ======================
print("=== Step 8/8: 绘制 Figure J 并保存结果 ===", flush=True)
best_prob = best_model.predict_proba(X_test)[:, 1]
best_fpr, best_tpr, _ = roc_curve(y_test, best_prob)
best_test_auc = auc(best_fpr, best_tpr)
best_test_ap = average_precision_score(y_test, best_prob)

plt.figure(figsize=(6, 6))
plt.plot(best_fpr, best_tpr, label=f"{best_name} AUC={best_test_auc:.3f}", color="#d88f91", linewidth=2)
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Final Best Model ROC on Independent Test Set")
plt.legend()
plt.tight_layout()
plt.savefig(f"{out_dir}/Figure_J_best_model_ROC.svg", dpi=300)
plt.close()
print("Figure J 已保存。", flush=True)


# ======================
# 保存测试集汇总
# ======================
test_summary_df = pd.DataFrame(test_summary).sort_values("test_auc", ascending=False)
test_summary_df.to_csv(f"{out_dir}/independent_test_summary.csv", index=False)

with open(f"{out_dir}/final_model_summary.txt", "w", encoding="utf-8") as f:
    f.write(f"Best model: {best_name}\n")
    f.write(f"Mean outer 10-fold CV AUC: {best_cv_auc:.4f}\n")
    f.write(f"Independent test AUC: {best_test_auc:.4f}\n")
    f.write(f"Independent test AP: {best_test_ap:.4f}\n")

print("\n==============================", flush=True)
print("全部运行完成！", flush=True)
print(f"最佳模型: {best_name}", flush=True)
print(f"独立测试集 AUC: {best_test_auc:.4f}", flush=True)
print(f"独立测试集 AP : {best_test_ap:.4f}", flush=True)
print(f"结果保存目录: {out_dir}", flush=True)
print("==============================", flush=True)

=== Step 1/8: 读取数据 ===
数据读取完成。
=== Step 2/8: 绘制 Figure F 单基因 ROC ===
Figure F 已保存。
=== Step 3/8: 划分训练集和测试集 ===
数据划分完成。
=== Step 4/8: 模型训练与评估开始 ===

[1/9] 正在处理模型: KNN ...
[1/9] KNN 完成
  最佳参数: {'model__n_neighbors': 9, 'model__weights': 'uniform'}
  内部5折最佳AUC: 0.6891
  外部10折平均AUC: 0.6817
  用时: 2.85 秒

[2/9] 正在处理模型: LDA ...
[2/9] LDA 完成
  最佳参数: {'model__solver': 'lsqr'}
  内部5折最佳AUC: 0.6955
  外部10折平均AUC: 0.6957
  用时: 1.35 秒

[3/9] 正在处理模型: LR ...
[3/9] LR 完成
  最佳参数: {'model__C': 0.01, 'model__solver': 'lbfgs'}
  内部5折最佳AUC: 0.6956
  外部10折平均AUC: 0.6957
  用时: 0.21 秒

[4/9] 正在处理模型: NB ...
[4/9] NB 完成
  最佳参数: {'model__var_smoothing': 1e-09}
  内部5折最佳AUC: 0.6875
  外部10折平均AUC: 0.6875
  用时: 0.13 秒

[5/9] 正在处理模型: RF ...
[5/9] RF 完成
  最佳参数: {'model__class_weight': 'balanced', 'model__max_depth': 3, 'model__min_samples_split': 5, 'model__n_estimators': 300}
  内部5折最佳AUC: 0.7138
  外部10折平均AUC: 0.7137
  用时: 5.69 秒

[6/9] 正在处理模型: RPART ...
[6/9] RPART 完成
  最佳参数: {'model__class_weight': None, 'model__max_dep